In [1]:

import pandas as pd
from dash import Dash, html, dcc, Input, Output
import plotly.express as px

In [2]:
df = pd.read_csv('C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\ecommerce_estatistica.csv')
print(df.head().to_string())

   Unnamed: 0                                                        Título  Nota  N_Avaliações  Desconto            Marca         Material               Gênero        Temporada                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       Review1                                                                                                                                                                                                                                                                                                                                         

In [4]:
lista_marcas = df['Marca'].unique()
options = [{'label': nivel, 'value': nivel} for nivel in lista_marcas]

def ecomerce(selecao_lista_marcas):
    filtro_df = df[df['Marca'].isin(selecao_lista_marcas)]
    
    fig1 = px.histogram(filtro_df, x='Preço', nbins=10, title='Distribuição de Preços')
    fig2 = px.pie(filtro_df, names='Gênero', color='Gênero', hole=0.1, color_discrete_sequence=px.colors.sequential.RdBu)
    fig3 = px.scatter(filtro_df, x='Desconto_MinMax', y='Preço_MinMax', size='Qtd_Vendidos_Cod', color='Qtd_Vendidos_Cod', size_max=60)
    fig3.update_layout(title='Desconto vs Preço por Quantidade Vendida')
    fig4 = px.density_heatmap(filtro_df, x='Gênero', y='Material', z='Qtd_Vendidos_Cod', color_continuous_scale='Viridis')
    fig5 = px.bar(filtro_df, x='Marca_Freq', y='Qtd_Vendidos_Cod', color='Gênero', barmode='group', color_discrete_sequence=px.colors.qualitative.Pastel)
    fig5.update_layout(
        title='Marca por QTD vendida e gênero',
        xaxis_title='Marca',
        yaxis_title='QTD vendida',
        legend_title='Gênero',
        plot_bgcolor='rgba(222, 255, 253, 1)',
        paper_bgcolor='rgba(186, 245, 241, 1)')
    fig6 = px.density_contour(filtro_df, x='Desconto', y='Preço_MinMax')
    fig6.update_layout(
        title='descontos dados',
        xaxis_title='Descontos',
        yaxis_title='Densidade'
    )
    return fig1, fig2, fig3, fig4, fig5, fig6
    
def cria_app():
    app = Dash(__name__)
    
    app.layout = html.Div([
        html.H1('Dashboard Ecomerce'),
        html.H2('Marcas'),
        
        dcc.Checklist(
            id='id_selecao_lista_marcas',
            options=options,
            value=[lista_marcas[0]],
        ),
        dcc.Graph(id='id_grafico_histograma'),
        dcc.Graph(id='id_grafico_pizza'),
        dcc.Graph(id='id_grafico_bolha'),
        dcc.Graph(id='id_grafico_calor'),
        dcc.Graph(id='id_grafico_barra'),
        dcc.Graph(id='id_grafico_densidade'),
    ])
    return app



if __name__ == '__main__':
    app = cria_app()
    
    @app.callback(
            [
        Output('id_grafico_histograma', 'figure'),
        Output('id_grafico_pizza', 'figure'),
        Output('id_grafico_bolha', 'figure'),
        Output('id_grafico_calor', 'figure'),
        Output('id_grafico_barra', 'figure'),
        Output('id_grafico_densidade', 'figure')
    ],
    [Input('id_selecao_lista_marcas', 'value')]
    )
    def atualizar_graficos(selecao_lista_marcas):
        fig1, fig2, fig3, fig4, fig5, fig6 = ecomerce(selecao_lista_marcas)
        return [fig1, fig2, fig3, fig4, fig5, fig6]
    app.run(debug=True, port=8051)  
    